# 상태도와 CALPHAD 실습

**Phase Diagram · CALPHAD · 상평형**

조성·온도·압력에 따라 안정한 상을 나타낸 지도와, 열역학 모델로 이를 계산하는 방법.

소재 분야에서 이해하기: 합금 조성에서 어떤 상이 먼저 생기는지 확인한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [pycalphad 상평형 계산 문서](https://pycalphad.org/)

## 1. 정용액 모형으로 혼합 자유에너지 그리기

혼합 엔탈피와 엔트로피의 경쟁으로 상분리가 생기는 것을 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

R = 8.314        # J/(mol K)

def free_energy(x, omega, temperature):
    """정용액(regular solution) 혼합 자유에너지 (J/mol). x 는 B 분율."""
    x = np.clip(x, 1e-9, 1 - 1e-9)
    return omega * x * (1 - x) + R * temperature * (x * np.log(x) + (1 - x) * np.log(1 - x))

omega = 20000.0      # 양수면 같은 원소끼리 뭉치려는 경향
x = np.linspace(0.001, 0.999, 500)
for temperature in (600, 900, 1200, 1500):
    plt.plot(x, free_energy(x, omega, temperature) / 1000, label='%d K' % temperature)
plt.xlabel('B fraction'); plt.ylabel('mixing free energy (kJ/mol)'); plt.legend(); plt.show()
print('임계온도 이론값 omega/2R = %.0f K' % (omega / (2 * R)))

## 2. 공통 접선으로 혼화 간극 구하기

In [ ]:
from scipy.optimize import brentq

def spinodal(omega, temperature):
    """d2G/dx2 = 0 인 두 조성 (스피노달)."""
    def curvature(x):
        return -2 * omega + R * temperature * (1 / x + 1 / (1 - x))
    try:
        return brentq(curvature, 1e-6, 0.5), brentq(curvature, 0.5, 1 - 1e-6)
    except ValueError:
        return None

def binodal(omega, temperature):
    """대칭 모형이므로 공통 접선은 x 와 1-x 에서 만납니다."""
    def condition(x):
        # dG/dx(x) == dG/dx(1-x) 는 대칭성으로 자동. 접선이 두 점을 잇는 조건:
        slope = (free_energy(1 - x, omega, temperature) - free_energy(x, omega, temperature)) / (1 - 2 * x)
        derivative = omega * (1 - 2 * x) + R * temperature * np.log(x / (1 - x))
        return derivative - slope
    try:
        x_low = brentq(condition, 1e-8, 0.4999)
        return x_low, 1 - x_low
    except ValueError:
        return None

temperatures = np.arange(300, 1300, 25.0)
bino, spino = [], []
for temperature in temperatures:
    bino.append(binodal(omega, temperature))
    spino.append(spinodal(omega, temperature))

plt.plot([b[0] for b in bino if b] + [b[1] for b in bino if b][::-1],
         list(temperatures[[b is not None for b in bino]]) + list(temperatures[[b is not None for b in bino]])[::-1],
         'b-', label='binodal (miscibility gap)')
plt.plot([s[0] for s in spino if s] + [s[1] for s in spino if s][::-1],
         list(temperatures[[s is not None for s in spino]]) + list(temperatures[[s is not None for s in spino]])[::-1],
         'r--', label='spinodal')
plt.xlabel('B fraction'); plt.ylabel('temperature (K)'); plt.xlim(0, 1); plt.legend(); plt.show()
print('돔 안쪽은 두 상으로 분리되는 영역입니다.')

## 3. 해석

실제 CALPHAD는 여러 상의 열역학 모델을 데이터로 맞추고, 상평형을 수치로 풉니다.
여기서는 가장 단순한 정용액 모형으로 상분리의 기원만 확인했습니다.
최근에는 이 열역학 모델의 파라미터 최적화에 머신러닝이 쓰입니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#phase-diagram)을 여세요.